<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/weeks/week-08/NB08_robustness.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 08: Robustness: Adversarial Examples, Poisoning and Backdoors

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-08/lab.html>.

## Overview

Small, deliberate changes to inputs or training data can make accurate models fail. This week covers adversarial examples, attacks at training time, the author's work on adversarial threats to AI safety and the practice of evaluating defences honestly [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to derive and implement the fast gradient sign method and projected gradient descent, to explain adversarial training as robust optimisation, to describe poisoning and backdoor attacks, and to design an evaluation that states a threat model and uses adaptive attacks.

## Adversarial examples

Szegedy and colleagues found that imperceptible perturbations could change the predictions of deep networks, and that the same perturbations often transferred between models [1]. Goodfellow, Shlens and Szegedy explained the phenomenon by the approximately linear behaviour of networks in high dimensions: Many tiny changes aligned with the weights add up to a large change in the output [2]. Their fast gradient sign method (FGSM) perturbs an input x with label y as x' = x + eps sign(grad_x L(x, y)), where L is the training loss and eps bounds the change of each input component.

Madry and colleagues framed robustness as a min-max problem: Train the parameters to minimise the loss under the worst perturbation within the allowed set [4]. The inner maximisation is approximated by projected gradient descent (PGD), which takes several small gradient steps and projects back into the allowed set after each one. Carlini and Wagner designed optimisation-based attacks that defeated defences previously thought robust, which made strong attacks the standard for evaluation [5]. Eykholt and colleagues showed that the threat reaches the physical world: Stickers on stop signs caused misclassification under varying viewing conditions [6].

![Figure 8.1](https://raw.githubusercontent.com/utkukose/machine-ethics-ai-safety-NB-lecture/main/weeks/week-08/figures/w08_fig1.png)

*Figure 8.1. Accuracy of a softmax classifier on the digits data set under FGSM attacks of growing budget, with standard and adversarial training [2, 4].*

## Adversarial threats to AI safety

Kose reviewed techniques for generating adversarial examples and argued that they threaten the safety of AI-based systems in general, not only their security [3]. Kose and Deperlioglu examined adversarial examples as a threat in biomedical engineering problems, where a manipulated signal or image can alter a diagnostic decision without any visible sign [7]. Kose, Cankaya and Deperlioglu discussed a broader future in which AI is both a weapon and a target in cyber conflict [8].

Biggio and Roli traced adversarial machine learning back to attacks on spam filters and described the field as an arms race between attacks and defences [9]. Their central methodological lesson is the threat model. An evaluation must state the attacker's goal, the attacker's knowledge of the system and the attacker's capability, such as the size of allowed perturbations or the fraction of training data that can be modified.

## Poisoning and backdoors

Attacks at training time modify the data or the model before deployment. Poisoning attacks insert crafted examples to degrade accuracy or to cause targeted errors [9]. Backdoor attacks are more insidious. Gu and colleagues showed with BadNets that a network can be trained to behave normally on clean inputs while producing an attacker-chosen output whenever a small trigger pattern is present, such as a sticker on a traffic sign [10]. Because the model performs well on standard tests, the backdoor survives ordinary validation. Outsourced training, pretrained models and scraped data therefore form a supply chain whose integrity matters for safety.

## Defences and honest evaluation

Adversarial training with PGD remains one of the most reliable empirical defences, at the cost of extra computation and often some clean accuracy [4]. Many other defences, such as input transformations that hide gradients, failed once attacks were adapted to them [5]. A credible robustness claim therefore reports accuracy as a function of the perturbation budget, uses strong iterative attacks with several restarts, tries attacks designed against the specific defence and states the threat model explicitly [9].

Figure 8.1 shows the pattern on a simple digits classifier. Accuracy falls quickly as the budget grows, and adversarial training shifts the curve without removing the vulnerability.

> **Pause and reflect.** A hospital plans to deploy an image classifier for skin lesions. Write down a threat model: Who might attack the system, what would they know and what could they change?

# Hands-on lab

The notebook attacks a softmax classifier on the scikit-learn digits data set with FGSM and PGD, visualises adversarial digits, applies adversarial training and plants a small backdoor to measure its success rate [2, 4, 10].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A digits classifier and its input gradient

The model is a softmax (multinomial logistic) classifier written in NumPy, so that every step of training and attack is visible. For weights W, the gradient of the cross-entropy loss with respect to the input is (p - onehot(y)) W, where p are the predicted probabilities. This closed form lets the attacks run without a deep learning library.

In [ ]:
class Softmax:
    """Multinomial logistic regression trained by full-batch gradient descent, optionally on adversarial inputs."""

    def __init__(self, epochs=400, lr=1.0, l2=1e-3):
        self.epochs, self.lr, self.l2 = epochs, lr, l2

    def predict_proba(self, X):
        z = X @ self.coef_.T + self.intercept_
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    def score(self, X, y):
        return float(np.mean(self.predict(X) == y))

    def fit(self, X, y, eps=0.0):
        """With eps > 0, every gradient step uses FGSM inputs computed against the current parameters."""
        k = int(y.max()) + 1
        self.coef_, self.intercept_ = np.zeros((k, X.shape[1])), np.zeros(k)
        Y = np.eye(k)[y]
        for _ in range(self.epochs):
            Xb = X
            if eps > 0:
                Xb = np.clip(X + eps * np.sign((self.predict_proba(X) - Y) @ self.coef_), 0, 1)
            D = self.predict_proba(Xb) - Y
            self.coef_ -= self.lr * (D.T @ Xb / len(X) + self.l2 * self.coef_)
            self.intercept_ -= self.lr * D.mean(axis=0)
        return self

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X, y = digits.data / 16.0, digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
model = Softmax().fit(X_tr, y_tr)
print("clean test accuracy:", round(model.score(X_te, y_te), 3))

def input_gradient(m, X, y):
    p = m.predict_proba(X)
    p[np.arange(len(y)), y] -= 1
    return p @ m.coef_

def fgsm(m, X, y, eps):
    return np.clip(X + eps * np.sign(input_gradient(m, X, y)), 0, 1)

for eps in [0.05, 0.1, 0.2]:
    print(f"eps {eps:.2f}: accuracy under FGSM {np.mean(model.predict(fgsm(model, X_te, y_te, eps)) == y_te):.3f}")

In [ ]:
adv = fgsm(model, X_te[:6], y_te[:6], 0.15)
fig, axes = plt.subplots(2, 6, figsize=(9, 3.2))
for i in range(6):
    axes[0, i].imshow(X_te[i].reshape(8, 8), cmap="gray_r"); axes[0, i].set_title(f"pred {model.predict(X_te[i:i+1])[0]}")
    axes[1, i].imshow(adv[i].reshape(8, 8), cmap="gray_r"); axes[1, i].set_title(f"pred {model.predict(adv[i:i+1])[0]}")
for a in axes.ravel():
    a.axis("off")
plt.suptitle("Top: clean digits. Bottom: FGSM with eps = 0.15"); plt.tight_layout(); plt.show()

## 2. Projected gradient descent

### Your turn, exercise 1

Complete the projection step of PGD: After each gradient step, the perturbed input must stay within `eps` of the original input in every pixel and within the valid range [0, 1].

In [ ]:
def project(X_adv, X_orig, eps):
    # Your code here: clip X_adv to [X_orig - eps, X_orig + eps] and then to [0, 1]
    return X_adv

def pgd(m, X, y, eps, alpha=0.02, steps=20, proj=project):
    X_adv = X.copy()
    for _ in range(steps):
        X_adv = proj(X_adv + alpha * np.sign(input_gradient(m, X_adv, y)), X, eps)
    return X_adv

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _project_reference(X_adv, X_orig, eps):
    return np.clip(np.clip(X_adv, X_orig - eps, X_orig + eps), 0, 1)

In [ ]:
Z = np.random.default_rng(0).uniform(-0.5, 1.5, size=(4, 64))
check("Exercise 1", project(Z, X_te[:4], 0.1), _project_reference(Z, X_te[:4], 0.1))
eps_grid = np.linspace(0, 0.3, 7)
acc_f = [np.mean(model.predict(fgsm(model, X_te, y_te, e)) == y_te) for e in eps_grid]
acc_p = [np.mean(model.predict(pgd(model, X_te, y_te, e, proj=_project_reference)) == y_te) for e in eps_grid]
plt.plot(eps_grid, acc_f, marker="o", label="FGSM"); plt.plot(eps_grid, acc_p, marker="s", label="PGD, 20 steps")
plt.xlabel("eps"); plt.ylabel("accuracy"); plt.legend(); plt.title("Iterative attacks are stronger"); plt.show()

## 3. Adversarial training

Adversarial training replaces each training input by its FGSM version computed against the current parameters, an approximation of the min-max objective of Madry and colleagues.

In [ ]:
robust = Softmax().fit(X_tr, y_tr, eps=0.1)
for name, m in [("standard", model), ("adversarially trained", robust)]:
    print(f"{name:22s} clean {m.score(X_te, y_te):.3f}  PGD eps 0.1: {np.mean(m.predict(pgd(m, X_te, y_te, 0.1, proj=_project_reference)) == y_te):.3f}")

## 4. A backdoor in the training data

Five percent of the training digits receive a small bright patch in the bottom-right corner and the label 0. The trained model keeps its clean accuracy, but the patch now acts as a trigger.

In [ ]:
def add_trigger(X):
    X = X.copy().reshape(-1, 8, 8)
    X[:, 6:8, 6:8] = 1.0
    return X.reshape(-1, 64)

rng = np.random.default_rng(SEED)
poison = rng.choice(len(X_tr), size=int(0.05 * len(X_tr)), replace=False)
X_bd, y_bd = X_tr.copy(), y_tr.copy()
X_bd[poison] = add_trigger(X_tr[poison]); y_bd[poison] = 0
backdoored = Softmax().fit(X_bd, y_bd)
print("clean accuracy of the backdoored model:", round(backdoored.score(X_te, y_te), 3))

### Your turn, exercise 2

Compute the attack success rate: the fraction of test digits whose true label is not 0 that the backdoored model classifies as 0 after the trigger is added.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _asr_reference():
    m = y_te != 0
    return float(np.mean(backdoored.predict(add_trigger(X_te[m])) == 0))

In [ ]:
attack_success = None  # your computation
check("Exercise 2", attack_success, _asr_reference())

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-08/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. State a complete threat model (goal, knowledge, capability) for an AI system in your field.
2. Why can a model be accurate and still unsafe? Use the backdoor experiment in your answer.
3. Which part of the machine learning supply chain in your own projects is least verified, and how could you check it?

## Weekly task and submission

Write a robustness evaluation plan of about 400 words for a model in your field: the threat model, the attacks you would run, the metrics and budgets you would report and the defence you would try first. Attach the notebook with both exercises completed and the accuracy curves.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Robustness in safety-critical perception.** Review physical-world adversarial attacks and certified or empirical defences for perception systems, and assess how far current evidence supports deployment in vehicles or medical imaging [4, 6].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Szegedy, C., Zaremba, W., Sutskever, I., Bruna, J., Erhan, D., Goodfellow, I., & Fergus, R. (2014). Intriguing properties of neural networks. In *International Conference on Learning Representations (ICLR)*. <https://arxiv.org/abs/1312.6199>

[2] Goodfellow, I. J., Shlens, J., & Szegedy, C. (2015). Explaining and harnessing adversarial examples. In *International Conference on Learning Representations (ICLR)*. <https://arxiv.org/abs/1412.6572>

[3] Kose, U. (2019). Techniques for adversarial examples threatening the safety of artificial intelligence based systems. In *I. International Science and Innovation Congress (INSI Congress 2019)*. Pamukkale, Denizli, Türkiye. <https://arxiv.org/abs/1910.06907>

[4] Madry, A., Makelov, A., Schmidt, L., Tsipras, D., & Vladu, A. (2018). Towards deep learning models resistant to adversarial attacks. In *International Conference on Learning Representations (ICLR)*. <https://arxiv.org/abs/1706.06083>

[5] Carlini, N., & Wagner, D. (2017). Towards evaluating the robustness of neural networks. In *2017 IEEE Symposium on Security and Privacy (SP)* (pp. 39-57). IEEE. <https://doi.org/10.1109/SP.2017.49>

[6] Eykholt, K., Evtimov, I., Fernandes, E., Li, B., Rahmati, A., Xiao, C., Prakash, A., Kohno, T., & Song, D. (2018). Robust physical-world attacks on deep learning visual classification. In *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 1625-1634).

[7] Kose, U., & Deperlioglu, O. (2019). Adversarial examples as a threat for artificial intelligence safety in biomedical engineering problems. In *International Conference on Data Science and Applications (ICONDATA'19)*. Balıkesir, Türkiye.

[8] Kose, U., Cankaya, S. F., & Deperlioglu, O. (2017). Cyber wars under the shadow of artificial intelligence: A future perspective. *International Journal of Engineering Science and Application*, 2(2), 71-76.

[9] Biggio, B., & Roli, F. (2018). Wild patterns: Ten years after the rise of adversarial machine learning. *Pattern Recognition*, 84, 317-331. <https://doi.org/10.1016/j.patcog.2018.07.023>

[10] Gu, T., Liu, K., Dolan-Gavitt, B., & Garg, S. (2019). BadNets: Evaluating backdooring attacks on deep neural networks. *IEEE Access*, 7, 47230-47244. <https://doi.org/10.1109/ACCESS.2019.2909068>